<a href="https://colab.research.google.com/github/Rems-dev1/lab-4-llm-decision-support/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Part 0: Repository and API-key setup**

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.
import os

# --- Google Colab (Secrets panel) ---
# We are using Colab, so we import userdata to safely grab our secret key!
from google.colab import userdata

# TODO: set API_KEY using ONE of the methods above.
# Make sure you have created a secret called "GROQ_API_KEY" in the left-hand key menu
API_KEY = userdata.get("GROQ_API_KEY")

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


**Section 1 — Talking to an LLM Programmatically**

In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.", temperature=0.7, max_tokens=500):
    # This sends our message to the AI
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    # We print this so we can see how much "brain power" the AI used
    print(f"[Tokens used: {response.usage.total_tokens}]")

    # This returns just the text answer so it is easy to read
    return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?
simple_question = "What is the capital city of Ghana?"
answer = ask_llm(simple_question)
print("\nAnswer from AI:")
print(answer)

[Tokens used: 59]

Answer from AI:
The capital city of Ghana is Accra.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
> 1. The `system` role is like setting the rules of a game; it tells the AI how to act (Example: "You are a grumpy math teacher"). The `user` role is what we actually ask it (Example: "What is 2 plus 2?").
> 2. A token is basically a piece of a word (like a syllable). Providers bill by token because generating each piece of a word requires computer power. A long answer costs them more energy than a short one!

**Part 1.2 — Temperature: the randomness dial**

In [5]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra."

# These lists will store the answers.
answers_0 = []
answers_12 = []

# Ask the question 5 times with temperature 0.0.
for i in range(5):
    answer = ask_llm(
        question,
        temperature=0.0,
        max_tokens=50
    )

    answers_0.append(answer)

# Ask the question 5 times with temperature 1.2.
for i in range(5):
    answer = ask_llm(
        question,
        temperature=1.2,
        max_tokens=50
    )

    answers_12.append(answer)


# Print the low-temperature answers.
print("Temperature = 0.0")

for i, answer in enumerate(answers_0, start=1):
    print(f"{i}. {answer}")


# Print the high-temperature answers.
print("\nTemperature = 1.2")

for i, answer in enumerate(answers_12, start=1):
    print(f"{i}. {answer}")

[Tokens used: 106]
[Tokens used: 106]
[Tokens used: 106]
[Tokens used: 106]
[Tokens used: 106]
[Tokens used: 106]
[Tokens used: 106]
[Tokens used: 106]
[Tokens used: 106]
[Tokens used: 106]
Temperature = 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **
2. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **
3. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **
4. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?*

> **Answer:**
> At temperature 0.0, the answers were usually very similar because the model was less random. At temperature 1.2, the answers were more varied and creative. For loan extraction and decision support, I would use a low temperature, such as 0.0, because we want stable and factual results. A higher temperature is better for creative ideas, not for important facts.